# Visualization 3 - Gender Scatter Plot

This notebook creates a simple visualization for the third mini-project question:

- How do naming patterns differ between boys and girls?
- For names used by both sexes, do popularity trends evolve in the same way?
- Can a name shift from mostly male to mostly female, or the reverse?

The chart uses a few **signature unisex names** and compares male and female births with a connected scatter plot.

Here, **each point is one year**, and the points are linked so you can follow how the name moves over time.

In [40]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()
alt.renderers.enable('default')

RendererRegistry.enable('default')

In [41]:
# Load and clean the dataset.
names = pd.read_csv('dpt2020.csv', sep=';')
names = names[(names['preusuel'] != '_PRENOMS_RARES') & (names['dpt'] != 'XX') & (names['annais'] != 'XXXX')].copy()
names['annais'] = names['annais'].astype(int)
names['nombre'] = names['nombre'].astype(int)

# National yearly totals by name and sex.
gender_year = names.groupby(['annais', 'preusuel', 'sexe'], as_index=False)['nombre'].sum()

# A few unisex names with clearly different trajectories.
selected_names = ['DOMINIQUE', 'CLAUDE', 'CHARLIE', 'LOU', 'SACHA']
plot_data = (
    gender_year[gender_year['preusuel'].isin(selected_names)]
    .pivot_table(index=['annais', 'preusuel'], columns='sexe', values='nombre', fill_value=0)
    .reset_index()
)
plot_data.columns = ['annais', 'preusuel', 'male_births', 'female_births']
plot_data = plot_data[(plot_data['annais'] >= 2000) & (plot_data['annais'] <= 2020)].copy()
plot_data['decade'] = (plot_data['annais'] // 10) * 10
plot_data['year_strength'] = (plot_data['annais'] - 2000) / 20
plot_data = plot_data.sort_values(['preusuel', 'annais']).copy()

# Keep only years where the name is meaningfully present for at least one sex.
plot_data = plot_data[(plot_data['male_births'] + plot_data['female_births']) >= 20].copy()

plot_data.head()

,annais,preusuel,male_births,female_births,decade,year_strength
309,2000,CHARLIE,87.0,12.0,2000,0.00
314,2001,CHARLIE,99.0,15.0,2000,0.05
319,2002,CHARLIE,53.0,10.0,2000,0.10
324,2003,CHARLIE,55.0,7.0,2000,0.15
329,2004,CHARLIE,63.0,11.0,2000,0.20


In [42]:
limits = max(plot_data['male_births'].max(), plot_data['female_births'].max())
reference = pd.DataFrame({'x': [0, limits], 'y': [0, limits]})

diag = alt.Chart(reference).mark_line(color='#999', strokeDash=[4, 4]).encode(
    x=alt.X('x:Q', title='Male births', scale=alt.Scale(domain=[0, limits], nice=False)),
    y=alt.Y('y:Q', title='Female births', scale=alt.Scale(domain=[0, limits], nice=False))
)

segments = plot_data.copy()
segments['male_births_next'] = segments.groupby('preusuel')['male_births'].shift(-1)
segments['female_births_next'] = segments.groupby('preusuel')['female_births'].shift(-1)
segments = segments.dropna(subset=['male_births_next', 'female_births_next']).copy()

name_palette = ['#d62828', '#1d4ed8', '#16a34a', '#f59e0b', '#7c3aed']

lines = alt.Chart(segments).mark_rule(strokeWidth=2.5).encode(
    x=alt.X('male_births:Q', title='Male births', scale=alt.Scale(domain=[0, limits], nice=False)),
    y=alt.Y('female_births:Q', title='Female births', scale=alt.Scale(domain=[0, limits], nice=False)),
    x2='male_births_next:Q',
    y2='female_births_next:Q',
    color=alt.Color('preusuel:N', title='Name', scale=alt.Scale(domain=selected_names, range=name_palette)),
    opacity=alt.Opacity('year_strength:Q', legend=None, scale=alt.Scale(domain=[0, 1], range=[0.2, 1]))
)

points = alt.Chart(plot_data).mark_circle(size=70).encode(
    x=alt.X('male_births:Q', title='Male births', scale=alt.Scale(domain=[0, limits], nice=False)),
    y=alt.Y('female_births:Q', title='Female births', scale=alt.Scale(domain=[0, limits], nice=False)),
    color=alt.Color('preusuel:N', title='Name', scale=alt.Scale(domain=selected_names, range=name_palette)),
    opacity=alt.Opacity('year_strength:Q', legend=None, scale=alt.Scale(domain=[0, 1], range=[0.25, 1])),
    tooltip=[
        alt.Tooltip('preusuel:N', title='Name'),
        alt.Tooltip('annais:Q', title='Year'),
        alt.Tooltip('male_births:Q', title='Male births'),
        alt.Tooltip('female_births:Q', title='Female births')
    ]
)

(diag + lines + points).properties(
    width=620,
    height=460,
    title='Shared Names Can Move Over Time Between Male and Female Usage (2000-2020)'
)

alt.LayerChart(...)

## Why this works for Visualization 3

### Advantages

- The path makes gender shifts over time easy to see for each shared name.
- The diagonal reference line clearly separates male-dominant and female-dominant usage.
- It answers the assignment question directly by showing whether the two sexes evolve consistently or not.
- Each point is one year for a shared name.
- The diagonal line shows the balance point: points above it are more female, points below it are more male.
- The connected path shows how a name moves over time between male and female usage.
- The visualization stays simple by focusing on a few **signature unisex names** rather than every shared name.


### Disadvantages

- The chart is based on a curated subset of shared names, not the full dataset.
- Without interaction, comparing many names at once could become cluttered.
- Some viewers may need a short explanation to understand the meaning of the diagonal line at first glance.
- The lines can overlap and make it hard to distinguish individual names if too many are included or the number of births is very small for some names, leading to a cluttered visualization.
- The color may not very clear to see the evolution of the years